# Day 4 – Sentence Embeddings


In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import pandas as pd
import itertools

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The cat sat on the sofa.",
    "A kitten is resting on the couch.",
    "I love eating pizza with extra cheese.",
    "Machine learning models learn patterns from data.",
    "AI systems are trained using large datasets."
]

embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)   # (5, 384): 5 sentences, 384 numbers each
print("First 5 numbers of sentence 1:", embeddings[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (5, 384)
First 5 numbers of sentence 1: [ 0.16549034 -0.08392975 -0.03161795  0.04167017 -0.03970971]


In [9]:
sim = util.cos_sim(embeddings[0], embeddings[1]).item()
print("Sentence A:", sentences[0])
print("Sentence B:", sentences[1])
print("Cosine similarity:", round(sim, 3))
print("Cosine distance:  ", round(1 - sim, 3))

Sentence A: The cat sat on the sofa.
Sentence B: A kitten is resting on the couch.
Cosine similarity: 0.696
Cosine distance:   0.304


In [10]:
sim_matrix = util.cos_sim(embeddings, embeddings).numpy()
labels = [f"S{i+1}" for i in range(len(sentences))]

df = pd.DataFrame(sim_matrix, index=labels, columns=labels).round(3)
display(df)

,S1,S2,S3,S4,S5
S1,1.000,0.696,-0.018,-0.043,-0.055
S2,0.696,1.000,-0.096,-0.084,-0.055
S3,-0.018,-0.096,1.000,0.033,-0.017
S4,-0.043,-0.084,0.033,1.000,0.518
S5,-0.055,-0.055,-0.017,0.518,1.000


In [11]:
pairs = []
for i, j in itertools.combinations(range(len(sentences)), 2):
    pairs.append((round(sim_matrix[i][j], 3), sentences[i], sentences[j]))

pairs.sort(reverse=True)
for score, a, b in pairs:
    print(f"{score}  |  {a}  <->  {b}")

0.6959999799728394  |  The cat sat on the sofa.  <->  A kitten is resting on the couch.
0.5180000066757202  |  Machine learning models learn patterns from data.  <->  AI systems are trained using large datasets.
0.032999999821186066  |  I love eating pizza with extra cheese.  <->  Machine learning models learn patterns from data.
-0.017000000923871994  |  I love eating pizza with extra cheese.  <->  AI systems are trained using large datasets.
-0.017999999225139618  |  The cat sat on the sofa.  <->  I love eating pizza with extra cheese.
-0.0430000014603138  |  The cat sat on the sofa.  <->  Machine learning models learn patterns from data.
-0.054999999701976776  |  The cat sat on the sofa.  <->  AI systems are trained using large datasets.
-0.054999999701976776  |  A kitten is resting on the couch.  <->  AI systems are trained using large datasets.
-0.08399999886751175  |  A kitten is resting on the couch.  <->  Machine learning models learn patterns from data.
-0.09600000083446503  |

In [12]:
tricky = [
    "The bank raised interest rates.",
    "We had a picnic on the river bank.",
    "The central bank increased its lending rate."
]
emb = model.encode(tricky)
print("bank (money) vs bank (river):", round(util.cos_sim(emb[0], emb[1]).item(), 3))
print("bank (money) vs central bank:", round(util.cos_sim(emb[0], emb[2]).item(), 3))

bank (money) vs bank (river): 0.268
bank (money) vs central bank: 0.75


## Reflection
The cat/kitten pair and the machine learning/AI pair scored highest, which matched my intuition. The pizza sentence scored lowest against everything else because its topic is unrelated. What surprised me most was the word "bank": the money sentence and the river sentence share the same word but scored only 0.268, while the money sentence and "central bank" scored 0.75. This showed me that embeddings capture meaning and context, not just matching words.